In [19]:
import os
import sys
import torch
import transformers

sys.path.append(os.path.dirname(os.getcwd()))

from transformers import AutoTokenizer, GenerationConfig

from model.configuration_sophie0 import Sophie0Config
from model.modeling_sophie0 import Sophie0ForCausalLM

base_path = os.path.dirname(os.getcwd())
tokenizer_path = os.path.join(base_path, "model/tokenizer")
model_path = os.path.join(base_path, "result/sft/finish/pytorch_model.bin")

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True, trust_remote_code=True, local_files_only=True)
model = Sophie0ForCausalLM(Sophie0Config())

state_dict = torch.load(model_path, map_location='cpu', weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>

In [20]:
device = "cuda:0"
dtype = torch.bfloat16
model = model.to(dtype=dtype, device=device)

In [23]:
generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=256,
    do_sample=True,
    top_k=20,
    top_p=0.8,
    temperature=0.8,
    num_beams=1,
    repeat_penalty=1.2,
    use_cache=True
)

prompt = [
    "<s><user>请问你是由谁训练研发的呢？</s>\n<s><bot>",
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>能否介绍一下中国的首都呢？</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>"
]
input_ids = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left").input_ids.to(device)

In [25]:
with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(input_ids, generate_config)

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <s><user>请问你是由谁训练研发的呢？</s>
<s><bot>作为AI语言模型，我没有个人情感，也没有情感。但是，我可以为你提供一些关于如何训练和训练神经网络的方法，但请注意，训练神经网络需要使用大量的数据和计算资源，因此我无法回答您的问题。</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pa

In [26]:
prompts = [
    "<s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$.</s>\n<s><bot>",
    "<s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$. You could think it step by step.</s>\n<s><bot>"
]
input_ids = tokenizer(prompts, return_tensors="pt", padding="longest", padding_side="left").input_ids.to(device)

generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=512,
    do_sample=True,
    top_k=20,
    top_p=0.8,
    temperature=0.8,
    num_beams=1,
    num_return_sequences=1,
    repeat_penalty=1.2,
    use_cache=True
)

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(input_ids, generate_config)

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <pad><pad><pad><pad><pad><pad><pad><pad><s><user><s><user>Given that $b$ is a multiple of $2373$, find the greatest common divisor of $b^2 + 13b + 40$ and $b + 5$.</s>
<s><bot>Since $b$ is a multiple of $2373$, we can write $b^2 + 13b + 40 = (2373 + 40)(2373 - 13b + 40)$. Expanding the right side, we get $2373^2 - 13b + 40 = 2373^2 - 15b + 40$. Since $b$ is a multiple of $2373$, we can write $b^2 + 13b + 40 = (2373 - 15b + 40)(2373 - 13b + 40)$. Expanding the right side, we get $2373^2 - 15b + 40 = (2373^2 - 15b + 40)(2373 - 13b + 40)$. Expanding the right side, we get $2373^2 - 15b + 40 = (2373 - 15b + 40)(2373 + 15b + 40)$. Expanding the right side, we get $2373^2 - 15b + 40 = (2373 - 15b + 40)(2373 + 15b + 40)$. Expanding the right side, we get $2373^2 - 15b + 40 = (2373 - 15b + 40)(2373 + 15b + 40)$. Expanding the right side, we get $2373^2 - 15b + 40 = (2373 - 15b + 40)(2373 + 15b + 40)$. Expanding the right side, we get $2373^2 - 15b + 40 = (2373 - 15b + 40)(2373 + 15b + 40)$.